<a href="https://colab.research.google.com/github/RajManish8340/gpt-shakespeare/blob/main/gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
# import libraries
import torch
import torch.nn as nn
from torch.nn import functional as F

In [41]:
# dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt' ,'r' , encoding='utf-8') as f:
  text = f.read()

print("number of characters",len(text))
print(text[:200])


--2026-08-08 08:34:12--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.4’

input.txt.4         100%[===================>]   1.06M  6.19MB/s    in 0.2s    

2026-08-08 08:34:13 (6.19 MB/s) - ‘input.txt.4’ saved [1115394/1115394]

number of characters 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [39]:
#hyper params
batch_size = 32
block_size = 64
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.0
device = "cuda" if torch.cuda.is_available() else "cpu"
#------------------

torch.manual_seed(1337) # for matching output form the video


In [42]:
#unique chars
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join((chars)))
print(vocab_size)

# string to integer and integer to string
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print(encode("hello"))
print(decode(encode("hello")))



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65
[46, 43, 50, 50, 53]
hello


In [43]:
# train and test splits
data = torch.tensor(encode(text) , dtype=torch.long)
print(data.shape , data.dtype)
print(f"first hundred tokens in data{data[:100]}")
n = int((0.9*len(data)))
train_data = data[:n]
val_data = data[n:]

torch.Size([1115394]) torch.int64
first hundred tokens in datatensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [44]:
# data loading
def get_batch(split):
  data = train_data if split == "train" else val_data
  ix = torch.randint(len(data) - block_size , (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device) , y.to(device)
  return x, y


In [45]:
@torch.no_grad
def estimate_loss():
  out = {}
  model.eval()
  for split in ["train", "val"]:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X,Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

class Head(nn.Module):
  """Single Head Attention """

  def __init__(self, head_size):
    super().__init__()
    self.key    = nn.Linear(n_embd, head_size, bias = False)
    self.query = nn.Linear(n_embd, head_size, bias = False)
    self.value  = nn.Linear(n_embd, head_size, bias = False)

    # every token only kows the history and the current , its 32 * 32 because
    # there are 32 possible combinations with 32 tokens (prev + self , no future)
    # and move to next token , values above diagonal = 0 .
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x) # B,T,C
    q = self.query(x) # B,T,C

    #affinities between keys and querries
    wei = q @ k.transpose(-2, -1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
    # lower trianguar matrix and the rest of the values which are above the
    # triangle which were zero now = -INF
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
    wei = F.softmax(wei, dim=-1) # (B, T, T)
    wei = self.dropout(wei)

    v = self.value(x) # (B, T, C)
    out = wei @ v #(B, T, T) @ (B, T, C) -> (B, T, C)
    return out

class MultiHeadAttention(nn.Module):
  """ Multi self attention Heads in parallel """

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out

class FeedForward(nn.Module):
  """ A simple linear leayer(nn.Linear) than non linearity(nn.ReLU)"""
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd), # 64 -> 4 * 64
        nn.ReLU(), # non linearity function
        nn.Linear(4*n_embd, n_embd), # 4 * 64 -> 64
        nn.Dropout(dropout)
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  """ Transformer block : comunication than computation """

  def __init__(self, n_embd, n_head):
    super().__init__()
    # head size 64/4 = 16 because after concatenation of the each head it will
    # become 64
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffd(self.ln2(x))
    return x

class BigramLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()
    # each token directly look off the logits for the next token form a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # final layerNorm
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    B, T = idx.shape

    # idx and target are both (B,T) taensor of integers
    tok_emb = self.token_embedding_table(idx)# (B,T,C)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(T,C)
    x = tok_emb + pos_emb # (B,T,C)
    x = self.blocks(x) # (B,T,C)
    x = self.ln_f(x) # (B,T,C)
    logits = self.lm_head(x) # (B,T,vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx is (B,T) array of indices in the current token
    for _ in range(max_new_tokens):
      # crop idx to last block_size token
      idx_cond = idx[:, -block_size:]
      # get predictions
      logits, losss = self(idx_cond)
      # focus only on the last token of the block not batch
      logits = logits[:, -1, :] # becomes (B,C)
      # apply softmax to get probabilities , -1 means we want to get prob of the
      # last dimensions which are the n_embd or weights
      probs = F.softmax(logits, dim=-1) # (B,C)
      # sample form distribution
      idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
      # append sampled index to running sequence
      idx = torch.cat((idx, idx_next), dim=1) #(B,T+1)
    return idx

model = BigramLanguageModel()
m = model.to(device)
# print number of parameters in model
print(sum(p.numel() for p in m.parameters())/1e6 , 'M parameters')

# create a pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

  # every once in a while evaluate the loss on train and val sets
  if iter % eval_interval == 0 or iter == max_iters - 1:
    losses = estimate_loss()
    print(f"step {iter}: train loss {losses["train"]:.4f}, val loss {losses["val"]:.4f}")

  # sample a batch data
  xb, yb = get_batch("train")

  # evaluate loss
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.816705 M parameters
step 0: train loss 4.3354, val loss 4.3307
step 100: train loss 2.4968, val loss 2.5072
step 200: train loss 2.3321, val loss 2.3344
step 300: train loss 2.1796, val loss 2.1964
step 400: train loss 2.0578, val loss 2.1158
step 500: train loss 1.9718, val loss 2.0547
step 600: train loss 1.8891, val loss 1.9931
step 700: train loss 1.8289, val loss 1.9447
step 800: train loss 1.7664, val loss 1.9039
step 900: train loss 1.7327, val loss 1.8889
step 1000: train loss 1.7075, val loss 1.8634
step 1100: train loss 1.6727, val loss 1.8197
step 1200: train loss 1.6401, val loss 1.7984
step 1300: train loss 1.6187, val loss 1.7868
step 1400: train loss 1.6028, val loss 1.7764
step 1500: train loss 1.5831, val loss 1.7576
step 1600: train loss 1.5748, val loss 1.7652
step 1700: train loss 1.5536, val loss 1.7398
step 1800: train loss 1.5435, val loss 1.7275
step 1900: train loss 1.5273, val loss 1.7180
step 2000: train loss 1.5180, val loss 1.7180
step 2100: train loss 1.